# Batch Ingestion

## Bronze Layer — Batch Data

This notebook ingests batch CSV files from the OneLake Files/batch folder into Delta tables in the Bronze layer.

Steps:
- Read the CSV file from the batch folder
- Use the first row as the header
- Infer the schema
- Inspect the data and schema
- Check for invalid column names
- Validate the source row count
- Add ingestion metadata
- Write the data to the Bronze Delta table
- Validate the Bronze row count

In [1]:
from pyspark.sql import functions as F

StatementMeta(, f7835a83-c299-4b2a-a6d9-e94dcf925f33, 4, Finished, Available, Finished, False)

## 1. Customers

In [3]:
df_customers = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("Files/batch/customers.csv")

StatementMeta(, f7835a83-c299-4b2a-a6d9-e94dcf925f33, 6, Finished, Available, Finished, False)

In [3]:
df_customers.show(10)

StatementMeta(, c1182055-467d-4088-8ec1-5f22f2d5a727, 5, Finished, Available, Finished, False)

+--------------------+--------------------+------------------------+---------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|  customer_city|customer_state|
+--------------------+--------------------+------------------------+---------------+--------------+
|c7432c6d237ffd6aa...|9a897ea48bf988012...|                    2971|      sao paulo|            SP|
|7f399d641e2e20644...|90436a67885a57f14...|                   38610|           unai|            MG|
|ba5642b730704dc0f...|4d8056f71519ae106...|                   88820|          icara|            SC|
|0f346a2cc84ebb2d5...|6117c9ef325108969...|                   25250|duque de caxias|            RJ|
|d393b9491df482cf4...|5caf3a2a5d1ef808e...|                   36955|          mutum|            MG|
|d9b4a26e122e830de...|99062accc4c9bbd3a...|                   79005|   campo grande|            MS|
|3d7ded9f88ad6b06b...|c7d1c1a9792b4c5b5...|                   13202|        jundiai|            SP|


In [4]:
df_customers.printSchema()

StatementMeta(, c1182055-467d-4088-8ec1-5f22f2d5a727, 6, Finished, Available, Finished, False)

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [4]:
df_customers.columns

StatementMeta(, f7835a83-c299-4b2a-a6d9-e94dcf925f33, 7, Finished, Available, Finished, False)

['customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state']

In [3]:
df_customers.count()

StatementMeta(, 401c6501-57b6-410b-aeb4-6771f776de5f, 10, Finished, Available, Finished, False)

10000

In [5]:
df_customers = (
    df_customers
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
)

StatementMeta(, c1182055-467d-4088-8ec1-5f22f2d5a727, 7, Finished, Available, Finished, False)

In [ ]:
df_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("1_bronze.bronze_customers")

StatementMeta(, c1182055-467d-4088-8ec1-5f22f2d5a727, 8, Finished, Available, Finished, False)

In [ ]:
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM 1_bronze.bronze_customers
""").show()

StatementMeta(, 401c6501-57b6-410b-aeb4-6771f776de5f, 11, Finished, Available, Finished, False)

+---------+
|row_count|
+---------+
|    10000|
+---------+



In [5]:
df_customers.count()

StatementMeta(, 401c6501-57b6-410b-aeb4-6771f776de5f, 16, Finished, Available, Finished, False)

10000

## 2. Order Items

In [5]:
df_order_items = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/batch/order_items.csv")
)

df_order_items.show(10)

df_order_items.printSchema()

StatementMeta(, f7835a83-c299-4b2a-a6d9-e94dcf925f33, 9, Finished, Available, Finished, False)

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|00063b381e2406b52...|            1|f177554ea93259a5b...|8602a61d680a10a82...|2018-07-31 17:30:39|  45.0|        12.98|
|000e63d38ae8c00bb...|            1|553e0e7590d3116a0...|1c129092bf23f28a5...|2018-03-29 20:07:49|  47.9|         8.88|
|00137e170939bba5a...|            1|672e757f331900b9d...|e59aa562b9f8076dd...|2017-11-30 06:30:55| 397.0|        24.65|
|002b430ff89b3a24c...|            1|cc47c0863559499f0...|7a67c85e85bb2ce85...|2017-06-26 22:10:14|199.99|        65.56|
|0030d783f979fbc59...|            1|ae27a5524edb2c8dc...|8e6cc767478edae94...|2017-11-29 23:13:38|  60.6|        17.67|
|00335f75ea6a4455b...|            1|e1bf

In [6]:
df_order_items.columns 

StatementMeta(, f7835a83-c299-4b2a-a6d9-e94dcf925f33, 11, Finished, Available, Finished, False)

['order_id',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value']

In [3]:
df_order_items.count()

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 7, Finished, Available, Finished, False)

11335

In [8]:
df_order_items = (
    df_order_items
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
)

StatementMeta(, c1182055-467d-4088-8ec1-5f22f2d5a727, 13, Finished, Available, Finished, False)

In [ ]:
df_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("1_bronze.bronze_order_items")

StatementMeta(, c1182055-467d-4088-8ec1-5f22f2d5a727, 14, Finished, Available, Finished, False)

In [ ]:
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM 1_bronze.bronze_order_items
""").show()

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 8, Finished, Available, Finished, False)

+---------+
|row_count|
+---------+
|    11335|
+---------+



In [14]:
df_order_items.count()

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 20, Finished, Available, Finished, False)

11335

## 3. Orders

In [8]:
df_orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/batch/orders.csv")
)

df_orders.show(10)

StatementMeta(, f7835a83-c299-4b2a-a6d9-e94dcf925f33, 14, Finished, Available, Finished, False)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|136cce7faa42fdb2c...|ed0271e0b7da060a3...|    invoiced|     2017-04-11 12:22:08|2017-04-13 13:25:17|                        NULL|                         NULL|          2017-05-09 00:00:00|
|dcb36b511fcac050b...|3b6828a50ffe54694...|   delivered|     2018-06-07 19:03:12|2018-06-12 23:31:02|         2018-06-11 14:54:00|          2018-06-21 15:34:32|          2018-07-04 00:00:00|
|85ce859fd6dc634de...|059f7fc5719c7da6c...|  

In [6]:
df_orders.printSchema()

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 12, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [9]:
df_orders.columns

StatementMeta(, f7835a83-c299-4b2a-a6d9-e94dcf925f33, 16, Finished, Available, Finished, False)

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

In [7]:
df_orders.count()

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 13, Finished, Available, Finished, False)

9946

In [10]:
df_orders = (
    df_orders
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
)

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 16, Finished, Available, Finished, False)

In [ ]:
df_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("1_bronze.bronze_orders")

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 17, Finished, Available, Finished, False)

In [ ]:
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM 1_bronze.bronze_orders
""").show()

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 18, Finished, Available, Finished, False)

+---------+
|row_count|
+---------+
|     9946|
+---------+



In [13]:
df_orders.count()

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 19, Finished, Available, Finished, False)

9946

## 4. Payments

In [11]:
df_payments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/batch/payments.csv")
)

df_payments.show(10)

StatementMeta(, f7835a83-c299-4b2a-a6d9-e94dcf925f33, 19, Finished, Available, Finished, False)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|2480f727e869fdeb3...|                 1| credit_card|                   1|        141.9|
|d0a945f85ba1074b6...|                 1| credit_card|                   2|        541.0|
|f45074ae38f2e01d9...|                 1| credit_card|                   1|        13.78|
|1dcf0c8cd36ffaf57...|                 1| credit_card|                   3|       157.15|
|00d8d65b666158b63...|                 1|      boleto|                   1|       130.88|
|6ffb3f1686bceae34...|                 1| credit_card|                   1|         32.0|
|c0db7d31ace61fc36...|                 1| credit_card|                   5|        65.71|
|1807cc736

In [16]:
df_payments.printSchema()

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 24, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



In [12]:
df_payments.columns

StatementMeta(, f7835a83-c299-4b2a-a6d9-e94dcf925f33, 21, Finished, Available, Finished, False)

['order_id',
 'payment_sequential',
 'payment_type',
 'payment_installments',
 'payment_value']

In [17]:
df_payments.count()

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 25, Finished, Available, Finished, False)

10356

In [18]:
df_payments = (
    df_payments
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
)

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 26, Finished, Available, Finished, False)

In [ ]:
df_payments.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("1_bronze.bronze_payments")

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 27, Finished, Available, Finished, False)

In [ ]:
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM 1_bronze.bronze_payments
""").show()

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 28, Finished, Available, Finished, False)

+---------+
|row_count|
+---------+
|    10356|
+---------+



In [21]:
df_payments.count()

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 29, Finished, Available, Finished, False)

10356

## 5. POS Sales Transactions

In [14]:
df_pos_sales_transactions = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/batch/pos_sales_transactions.csv")
)

df_pos_sales_transactions.show(10)

StatementMeta(, f7835a83-c299-4b2a-a6d9-e94dcf925f33, 24, Finished, Available, Finished, False)

+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
| 532657|    21314|SMALL GLASS HEART...|      12|2010-11-14 11:10:00|  2.1|    14562.0|United Kingdom|
| 563214|    22383|LUNCH BAG SUKI DE...|       2|2011-08-14 12:56:00| 1.65|    16370.0|United Kingdom|
| 507597|    22561|WOODEN SCHOOL COL...|      12|2010-05-10 13:21:00| 1.65|    17700.0|United Kingdom|
| 491634|    21588|RETRO SPOT GIANT ...|       1|2009-12-11 15:40:00| 2.55|    17841.0|United Kingdom|
| 496007|   85232B|SET/3 RUSSIAN DOL...|       3|2010-01-28 12:32:00| 4.95|    15203.0|United Kingdom|
| 539041|    21832|CHOCOLATE CALCULATOR|       4|2010-12-15 15:34:00| 1.65|    15456.0|United Kingdom|
| 575905|    22089|PAPER BUNTING VIN...|       6|2011-11-11 15:49:00| 2.9

In [27]:
df_pos_sales_transactions = df_pos_sales_transactions.withColumnRenamed(
    "Customer ID",
    "customer_ID"
)

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 37, Finished, Available, Finished, False)

In [23]:
df_pos_sales_transactions.printSchema()

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 33, Finished, Available, Finished, False)

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)



In [15]:
df_pos_sales_transactions.columns 

StatementMeta(, f7835a83-c299-4b2a-a6d9-e94dcf925f33, 26, Finished, Available, Finished, False)

['Invoice',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'Price',
 'Customer ID',
 'Country']

In [24]:
df_pos_sales_transactions.count()

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 34, Finished, Available, Finished, False)

75000

In [25]:
df_pos_sales_transactions = (
    df_pos_sales_transactions
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
)

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 35, Finished, Available, Finished, False)

In [ ]:
df_pos_sales_transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("1_bronze.bronze_pos_sales_transactions")

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 38, Finished, Available, Finished, False)

In [ ]:
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM 1_bronze.bronze_pos_sales_transactions
""").show()

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 39, Finished, Available, Finished, False)

+---------+
|row_count|
+---------+
|    75000|
+---------+



In [33]:
df_pos_sales_transactions.count()

StatementMeta(, a956dee8-98c9-40c5-bc96-664de9adbfe9, 44, Finished, Available, Finished, False)

75000

## 6. Products

In [1]:
df_products = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/batch/products.csv")
)

df_products.show(10)

StatementMeta(, a43cf145-15f2-46a1-aed9-528664880c4c, 9, Finished, Available, Finished, False)

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|d03bd02af9fff4b98...|     moveis_decoracao|               56.0|                     209.0|               1.0|          1800.0|             40.0|             10.0|            30.0|
|1eba879220bd0981a...|    moveis_escritorio|               57.0|                     476.0|               1.0|          8950.0|             52.0|             51.0|            17.0|
|5370b82a213393979...|                bebes|               52.0|                     708.0|    

In [2]:
df_products.printSchema()

StatementMeta(, a43cf145-15f2-46a1-aed9-528664880c4c, 11, Finished, Available, Finished, False)

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: double (nullable = true)
 |-- product_description_lenght: double (nullable = true)
 |-- product_photos_qty: double (nullable = true)
 |-- product_weight_g: double (nullable = true)
 |-- product_length_cm: double (nullable = true)
 |-- product_height_cm: double (nullable = true)
 |-- product_width_cm: double (nullable = true)



In [3]:
df_products.columns

StatementMeta(, a43cf145-15f2-46a1-aed9-528664880c4c, 12, Finished, Available, Finished, False)

['product_id',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm']

In [4]:
df_products.count()

StatementMeta(, a43cf145-15f2-46a1-aed9-528664880c4c, 13, Finished, Available, Finished, False)

6827

In [7]:
df_products = (
    df_products
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
)

StatementMeta(, a43cf145-15f2-46a1-aed9-528664880c4c, 16, Finished, Available, Finished, False)

In [ ]:
df_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("1_bronze.bronze_products")

StatementMeta(, a43cf145-15f2-46a1-aed9-528664880c4c, 17, Finished, Available, Finished, False)

In [ ]:
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM 1_bronze.bronze_products
""").show()

StatementMeta(, a43cf145-15f2-46a1-aed9-528664880c4c, 18, Finished, Available, Finished, False)

+---------+
|row_count|
+---------+
|     6827|
+---------+



In [10]:
df_products.count()

StatementMeta(, a43cf145-15f2-46a1-aed9-528664880c4c, 19, Finished, Available, Finished, False)

6827

## 7. Reviews

In [3]:
df_reviews = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/batch/reviews.csv")
)

df_reviews.show(10)

StatementMeta(, ee6196e5-595f-4cf0-84b6-0d79a1dd6ac9, 6, Finished, Available, Finished, False)

+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|7c6400515c67679fb...|c31a859e34e3adac2...|           5|                NULL|                  NULL| 2018-08-14 00:00:00|    2018-08-14 21:36:06|
|4b49719c8a200003f...|9d6f15f95d01e79bd...|           4|                NULL|  Mas um pouco ,tra...|                NULL|                   NULL|
|,2018-02-16 00:00...|                NULL|        NULL|                NULL|                  NULL|                NULL|                   NULL|
|3948b09f7c818e2d8...|e51478e7e277a8374...|           5|     Super recomendo|  Vendedor confiáve...| 2018-05-23 00:00:00|   

In [12]:
df_reviews.printSchema()

StatementMeta(, a43cf145-15f2-46a1-aed9-528664880c4c, 23, Finished, Available, Finished, False)

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



In [13]:
df_reviews.columns

StatementMeta(, a43cf145-15f2-46a1-aed9-528664880c4c, 24, Finished, Available, Finished, False)

['review_id',
 'order_id',
 'review_score',
 'review_comment_title',
 'review_comment_message',
 'review_creation_date',
 'review_answer_timestamp']

In [14]:
df_reviews.count()

StatementMeta(, a43cf145-15f2-46a1-aed9-528664880c4c, 25, Finished, Available, Finished, False)

10362

In [4]:
df_reviews = (
    df_reviews
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
)

StatementMeta(, ee6196e5-595f-4cf0-84b6-0d79a1dd6ac9, 8, Finished, Available, Finished, False)

In [ ]:
df_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("1_bronze.bronze_reviews")

StatementMeta(, ee6196e5-595f-4cf0-84b6-0d79a1dd6ac9, 9, Finished, Available, Finished, False)

In [ ]:
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM 1_bronze.bronze_reviews
""").show()

StatementMeta(, ee6196e5-595f-4cf0-84b6-0d79a1dd6ac9, 10, Finished, Available, Finished, False)

+---------+
|row_count|
+---------+
|    10362|
+---------+



In [7]:
df_reviews.count()

StatementMeta(, ee6196e5-595f-4cf0-84b6-0d79a1dd6ac9, 11, Finished, Available, Finished, False)

10362

## 8. Sellers

In [8]:
df_sellers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/batch/sellers.csv")
)

df_sellers.show(10)

StatementMeta(, ee6196e5-595f-4cf0-84b6-0d79a1dd6ac9, 13, Finished, Available, Finished, False)

+--------------------+----------------------+------------+------------+
|           seller_id|seller_zip_code_prefix| seller_city|seller_state|
+--------------------+----------------------+------------+------------+
|3442f8959a84dea7e...|                 13023|    campinas|          SP|
|d1b65fc7debc3361e...|                 13844|  mogi guacu|          SP|
|e49c26c3edfa46d22...|                 55325|      brejao|          PE|
|1b938a7ec6ac5061a...|                 16304|   penapolis|          SP|
|768a86e36ad6aae3d...|                  1529|   sao paulo|          SP|
|ccc4bbb5f32a6ab2b...|                 80310|    curitiba|          PR|
|8bd0f31cf0a614c65...|                  1222|   sao paulo|          SP|
|7b8e8ec35bad4b0ef...|                 88705|     tubarao|          SC|
|e38db885400cd35c7...|                 70740|    brasilia|          DF|
|d2e753bb80b7d4faa...|                 45810|porto seguro|          BA|
+--------------------+----------------------+------------+------

In [9]:
df_sellers.printSchema()

StatementMeta(, ee6196e5-595f-4cf0-84b6-0d79a1dd6ac9, 15, Finished, Available, Finished, False)

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



In [10]:
df_sellers.columns

StatementMeta(, ee6196e5-595f-4cf0-84b6-0d79a1dd6ac9, 16, Finished, Available, Finished, False)

['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

In [11]:
df_sellers.count()

StatementMeta(, ee6196e5-595f-4cf0-84b6-0d79a1dd6ac9, 17, Finished, Available, Finished, False)

1653

In [12]:
df_sellers = (
    df_sellers
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
)

StatementMeta(, ee6196e5-595f-4cf0-84b6-0d79a1dd6ac9, 18, Finished, Available, Finished, False)

In [ ]:
df_sellers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("1_bronze.bronze_sellers")

StatementMeta(, ee6196e5-595f-4cf0-84b6-0d79a1dd6ac9, 19, Finished, Available, Finished, False)

In [ ]:
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM 1_bronze.bronze_sellers
""").show()

StatementMeta(, ee6196e5-595f-4cf0-84b6-0d79a1dd6ac9, 20, Finished, Available, Finished, False)

+---------+
|row_count|
+---------+
|     1653|
+---------+



In [15]:
df_sellers.count()

StatementMeta(, ee6196e5-595f-4cf0-84b6-0d79a1dd6ac9, 21, Finished, Available, Finished, False)

1653

## Bronze Layer — Reference Data

The reference datasets are ingested from the `Files/reference` folder into the Bronze layer as Delta tables.

Each reference dataset follows the same Bronze ingestion pattern:

1. Read the CSV file from the `reference` folder.
2. Use the first row as the column header.
3. Infer the data types using Spark.
4. Inspect the data using `show()`.
5. Inspect the schema using `printSchema()`.
6. Check the column names for invalid characters.
7. Validate the source row count.
8. Add ingestion metadata:
   - `_source_file`
   - `_ingestion_timestamp`
   - `_ingestion_date`
9. Write the DataFrame as a Delta table in the Bronze layer.
10. Validate the Bronze table row count against the source DataFrame count.

In [1]:
df_employees = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/reference/employees.csv")
)

df_employees.show(10)

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 7, Finished, Available, Finished, False)

+-----------+----------+---------+----------+----------+--------+----------+
|employee_id|first_name|last_name|department| job_title|store_id| hire_date|
+-----------+----------+---------+----------+----------+--------+----------+
|  EMP_00001|   Staff_1|     LN_1|     Sales|Specialist| STR_008|2024-01-15|
|  EMP_00002|   Staff_2|     LN_2|Operations| Associate| STR_047|2021-04-02|
|  EMP_00003|   Staff_3|     LN_3|     Sales| Associate| STR_037|2022-11-04|
|  EMP_00004|   Staff_4|     LN_4|     Sales| Associate| STR_003|2022-12-15|
|  EMP_00005|   Staff_5|     LN_5|Operations|Specialist| STR_033|2023-07-25|
|  EMP_00006|   Staff_6|     LN_6|     Sales| Associate| STR_028|2021-08-30|
|  EMP_00007|   Staff_7|     LN_7|Operations|Supervisor| STR_047|2021-01-02|
|  EMP_00008|   Staff_8|     LN_8|     Sales| Associate| STR_008|2021-08-13|
|  EMP_00009|   Staff_9|     LN_9|     Sales| Associate| STR_034|2023-03-12|
|  EMP_00010|  Staff_10|    LN_10| Inventory|Specialist| STR_035|2022-08-12|

In [2]:
df_employees.printSchema() 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 9, Finished, Available, Finished, False)

root
 |-- employee_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- hire_date: date (nullable = true)



In [3]:
df_employees.columns 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 10, Finished, Available, Finished, False)

['employee_id',
 'first_name',
 'last_name',
 'department',
 'job_title',
 'store_id',
 'hire_date']

In [4]:
df_employees.count() 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 11, Finished, Available, Finished, False)

150

In [7]:
df_employees = (
    df_employees
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
)

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 14, Finished, Available, Finished, False)

In [ ]:
df_employees.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("1_bronze.bronze_employees")

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 15, Finished, Available, Finished, False)

In [ ]:
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM 1_bronze.bronze_employees
""").show() 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 16, Finished, Available, Finished, False)

+---------+
|row_count|
+---------+
|      150|
+---------+



In [10]:
df_employees.count()

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 17, Finished, Available, Finished, False)

150

In [11]:
df_inventory_snapshots = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/reference/inventory_snapshots.csv")
)

df_inventory_snapshots.show(10)

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 19, Finished, Available, Finished, False)

+-------------+--------+--------------------+-------------+------------------+--------------+
|snapshot_date|store_id|          product_id|stock_on_hand|safety_stock_level|  stock_status|
+-------------+--------+--------------------+-------------+------------------+--------------+
|   2026-08-08| STR_006|eba7488e1c67729f0...|           32|                15|Healthy Bucket|
|   2026-08-08| STR_006|e7593e3c84b3302e1...|           19|                15|Healthy Bucket|
|   2026-08-08| STR_006|9cd17b1cc8db0de6b...|          139|                15|Healthy Bucket|
|   2026-08-08| STR_006|5862da8a8a0888130...|          100|                15|Healthy Bucket|
|   2026-08-08| STR_006|0ecade74f3c3fb56a...|           22|                15|Healthy Bucket|
|   2026-08-08| STR_006|2afd040779d1ce629...|          111|                15|Healthy Bucket|
|   2026-08-08| STR_006|de72026142ffc78ea...|           13|                15| Reorder Alert|
|   2026-08-08| STR_006|cb81df0e3ccece253...|           90| 

In [12]:
df_inventory_snapshots.printSchema() 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 21, Finished, Available, Finished, False)

root
 |-- snapshot_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- stock_on_hand: integer (nullable = true)
 |-- safety_stock_level: integer (nullable = true)
 |-- stock_status: string (nullable = true)



In [13]:
df_inventory_snapshots.columns 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 22, Finished, Available, Finished, False)

['snapshot_date',
 'store_id',
 'product_id',
 'stock_on_hand',
 'safety_stock_level',
 'stock_status']

In [14]:
df_inventory_snapshots.count() 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 23, Finished, Available, Finished, False)

1500

In [15]:
df_inventory_snapshots = (
    df_inventory_snapshots
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
)

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 24, Finished, Available, Finished, False)

In [ ]:
df_inventory_snapshots.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("1_bronze.bronze_inventory_snapshots") 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 25, Finished, Available, Finished, False)

In [ ]:
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM 1_bronze.bronze_inventory_snapshots
""").show() 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 26, Finished, Available, Finished, False)

+---------+
|row_count|
+---------+
|     1500|
+---------+



In [18]:
df_inventory_snapshots.count() 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 27, Finished, Available, Finished, False)

1500

In [19]:
df_promotions = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/reference/promotions.csv")
)

df_promotions.show(10) 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 30, Finished, Available, Finished, False)

+------------+--------------------+-------------------+-----------------+------------+
|promotion_id|       campaign_name|discount_percentage|marketing_channel|is_stackable|
+------------+--------------------+-------------------+-----------------+------------+
|     PRM_001|Seasonal Boost Wa...|                0.2|      Email Blast|       false|
|     PRM_002|Seasonal Boost Wa...|                0.3|      Email Blast|       false|
|     PRM_003|Seasonal Boost Wa...|               0.05|      Email Blast|       false|
|     PRM_004|Seasonal Boost Wa...|                0.1|     Social Media|       false|
|     PRM_005|Seasonal Boost Wa...|               0.05|     Social Media|       false|
|     PRM_006|Seasonal Boost Wa...|               0.05|  In-Store Banner|       false|
|     PRM_007|Seasonal Boost Wa...|                0.2|  In-Store Banner|       false|
|     PRM_008|Seasonal Boost Wa...|                0.1|Push Notification|       false|
|     PRM_009|Seasonal Boost Wa...|        

In [20]:
df_promotions.printSchema() 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 32, Finished, Available, Finished, False)

root
 |-- promotion_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- discount_percentage: double (nullable = true)
 |-- marketing_channel: string (nullable = true)
 |-- is_stackable: boolean (nullable = true)



In [21]:
df_promotions.columns 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 33, Finished, Available, Finished, False)

['promotion_id',
 'campaign_name',
 'discount_percentage',
 'marketing_channel',
 'is_stackable']

In [22]:
df_promotions.count() 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 34, Finished, Available, Finished, False)

30

In [23]:
df_promotions = (
    df_promotions
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
) 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 35, Finished, Available, Finished, False)

In [ ]:
df_promotions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("1_bronze.bronze_promotions") 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 36, Finished, Available, Finished, False)

In [ ]:
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM 1_bronze.bronze_promotions
""").show() 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 37, Finished, Available, Finished, False)

+---------+
|row_count|
+---------+
|       30|
+---------+



In [26]:
df_promotions.count() 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 38, Finished, Available, Finished, False)

30

In [27]:
df_stores = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/reference/stores.csv")
)

df_stores.show(10)

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 40, Finished, Available, Finished, False)

+--------+--------------------+----------+--------------+------------+---------+
|store_id|          store_name|store_type|          city|      region|is_active|
+--------+--------------------+----------+--------------+------------+---------+
| STR_001|Retail Hub Brasíl...|  Standard|      Brasília|Central-West|     true|
| STR_002|Retail Hub Salvad...|  Standard|      Salvador|   Northeast|     true|
| STR_003|Retail Hub Belo H...|  Flagship|Belo Horizonte|   Southeast|     true|
| STR_004|Retail Hub Salvad...|  Standard|      Salvador|   Northeast|     true|
| STR_005|Retail Hub Salvad...|  Standard|      Salvador|   Northeast|     true|
| STR_006|Retail Hub Rio de...|   Express|Rio de Janeiro|   Southeast|     true|
| STR_007|Retail Hub Belo H...|   Express|Belo Horizonte|   Southeast|    false|
| STR_008|Retail Hub Belo H...|   Express|Belo Horizonte|   Southeast|     true|
| STR_009|Retail Hub Belo H...|  Standard|Belo Horizonte|   Southeast|     true|
| STR_010|Retail Hub Salvad.

In [28]:
df_stores.printSchema() 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 42, Finished, Available, Finished, False)

root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- is_active: boolean (nullable = true)



In [29]:
df_stores.columns 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 43, Finished, Available, Finished, False)

['store_id', 'store_name', 'store_type', 'city', 'region', 'is_active']

In [30]:
df_stores.count() 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 44, Finished, Available, Finished, False)

50

In [31]:
df_stores = (
    df_stores
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
) 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 45, Finished, Available, Finished, False)

In [ ]:
df_stores.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("1_bronze.bronze_stores")

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 46, Finished, Available, Finished, False)

In [ ]:
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM 1_bronze.bronze_stores
""").show() 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 47, Finished, Available, Finished, False)

+---------+
|row_count|
+---------+
|       50|
+---------+



In [34]:
df_stores.count() 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 48, Finished, Available, Finished, False)

50

In [35]:
df_suppliers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/reference/suppliers.csv")
)

df_suppliers.show(10)

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 50, Finished, Available, Finished, False)

+-----------+--------------------+----------------+------------------+-------------+
|supplier_id|       supplier_name|primary_category|     supplier_tier|      country|
+-----------+--------------------+----------------+------------------+-------------+
|   SUP_0001|Global Alpha Corp...|   Home & Living| Tier 3 - Standard|       Brazil|
|   SUP_0002|Global Alpha Corp...|            Toys|Tier 1 - Strategic|       Brazil|
|   SUP_0003|Global Alpha Corp...|     Electronics|Tier 2 - Preferred|United States|
|   SUP_0004|Global Alpha Corp...| Health & Beauty|Tier 2 - Preferred|       Brazil|
|   SUP_0005|Global Alpha Corp...|     Electronics| Tier 3 - Standard|        China|
|   SUP_0006|Global Alpha Corp...|            Toys|Tier 2 - Preferred|      Germany|
|   SUP_0007|Global Alpha Corp...|     Electronics|Tier 2 - Preferred|       Brazil|
|   SUP_0008|Global Alpha Corp...|         Fashion|Tier 1 - Strategic|      Germany|
|   SUP_0009|Global Alpha Corp...| Health & Beauty|Tier 2 - Prefe

In [36]:
df_suppliers.printSchema() 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 52, Finished, Available, Finished, False)

root
 |-- supplier_id: string (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- primary_category: string (nullable = true)
 |-- supplier_tier: string (nullable = true)
 |-- country: string (nullable = true)



In [37]:
df_suppliers.columns 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 53, Finished, Available, Finished, False)

['supplier_id',
 'supplier_name',
 'primary_category',
 'supplier_tier',
 'country']

In [38]:
df_suppliers.count() 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 54, Finished, Available, Finished, False)

250

In [39]:
df_suppliers = (
    df_suppliers
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
) 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 55, Finished, Available, Finished, False)

In [ ]:
df_suppliers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("1_bronze.bronze_suppliers") 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 56, Finished, Available, Finished, False)

In [ ]:
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM 1_bronze.bronze_suppliers
""").show()

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 57, Finished, Available, Finished, False)

+---------+
|row_count|
+---------+
|      250|
+---------+



In [42]:
df_suppliers.count() 

StatementMeta(, 95f680df-b341-46c0-8883-b98cd70ce088, 58, Finished, Available, Finished, False)

250